# Structural Sources of Real-vs-Synthetic Separability

**Research question:** Are real-vs-synthetic RF-AUC results driven by a reproducible subset of distorted feature relationships?

The main notebook establishes two observations: (1) a random forest can distinguish real from synthetic rows, and (2) Graphical Lasso finds preserved, lost, invented, reversed, or changed conditional relationships. This notebook tests whether those observations are functionally connected.

## Experimental logic

For each repeated split:

1. Use discovery rows only to fit real and synthetic Graphical Lasso models.
2. Rank structural errors without using the RF discriminator.
3. Repair the top 1, 5, 10, or 20 relationships in held-out synthetic rows using a Gaussian-copula rank transform.
4. Preserve the exact empirical marginal values of every repaired synthetic feature.
5. Train fresh RF discriminators on held-out real and repaired synthetic rows.
6. Compare targeted repair with random-pair, already-preserved-edge, and marginal-only controls.

**Support for the hypothesis:** targeted repairs reduce held-out AUC more than controls, the reduction grows with dose, and the same relationships recur across discovery splits.

**Evidence against the hypothesis:** targeted repairs behave like random repairs, AUC does not fall, or selected relationships are unstable across splits.

In [ ]:
from pathlib import Path
import pickle
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parents[1]
pkg_root = repo_root / "data_synthesis"
if str(pkg_root) not in sys.path:
    sys.path.insert(0, str(pkg_root))

from src.revision import cache as revision_cache
from src.revision import config as revision_config
from src.revision import data_io
from src.revision import figure2_auc
from src.revision.common import sample_synthetic
from src.revision.figure4_graphical_lasso import FIGURE4_ALPHAS
from src.revision.structural_repair import (
    ERROR_CATEGORIES,
    build_edge_discrepancy_table,
    plot_edge_discrepancy_map,
    plot_repair_dose_response,
    run_graph_guided_repair_experiment,
    summarize_repair_results,
)


In [ ]:
# Start with one dataset-generator pair so every diagnostic remains inspectable.
DATASET = "HIV"
METHOD = "CVAE"
DOSES = (0, 1, 5, 10, 20)
SPLIT_REPEATS = 3
AUC_REPEATS = 5
DISCOVERY_FRACTION = 0.5
CHANGED_PARTIAL_THRESHOLD = 0.05
REPAIR_STRENGTH = 1.0
FORCE_RECOMPUTE = False

CACHE_VERSION = 1
cache_name = f"structural_repair_v{CACHE_VERSION}_{DATASET}_{METHOD}".lower().replace(" ", "_") + ".pkl"
experiment_cache = revision_config.CACHE_DIR / cache_name
experiment_cache


## Reference 1: RF origin separability

This is the same cached RF-AUC measurement used by `make_main_figures_revision.ipynb`. It defines the functional outcome that the repair experiment tries to reduce.

In [ ]:
auc_runs = revision_cache.get_auc_runs(force=False)
selected_auc = auc_runs[(auc_runs["dataset"] == DATASET) & (auc_runs["method"] == METHOD)]
display(selected_auc["separability_auc"].describe().to_frame("RF origin AUC"))

fig, ax = plt.subplots(figsize=(7.2, 4.2))
figure2_auc._plot_auc_violin_panel(ax, auc_runs, DATASET, "A")
ax.set_title(f"{DATASET}: baseline real-vs-synthetic separability", loc="left", weight="bold")
plt.show()


In [ ]:
datasets, dataset_summary = data_io.initialize_datasets()
data = datasets[DATASET]
X_real = np.asarray(data["X"], dtype=np.float64)
y_real = np.asarray(data["y"], dtype=int)
feature_names = list(data["feature_names"])

# This uses the same generator, seed, class counts, and final run settings as the main notebook.
X_syn, y_syn = sample_synthetic(DATASET, data, METHOD, seed=revision_config.SEED)
X_syn = np.asarray(X_syn, dtype=np.float64)
y_syn = np.asarray(y_syn, dtype=int)

display(dataset_summary)
print(f"Selected comparison: {DATASET} / {METHOD}; real={X_real.shape}, synthetic={X_syn.shape}")


## Reference 2: Graphical Lasso relationship errors

The edge definitions and fixed dataset alpha are imported from the Figure 4 implementation. The full-data map below is descriptive context only. Candidate repairs in the experiment are rediscovered independently inside each discovery split.

In [ ]:
edge_table_full = build_edge_discrepancy_table(
    X_real,
    X_syn,
    feature_names=feature_names,
    alpha=FIGURE4_ALPHAS[DATASET],
    changed_threshold=CHANGED_PARTIAL_THRESHOLD,
)

display(edge_table_full["category"].value_counts().rename_axis("edge status").to_frame("count"))
display(edge_table_full[edge_table_full["category"].isin(ERROR_CATEGORIES)].head(20))
plot_edge_discrepancy_map(
    edge_table_full,
    feature_names,
    title=f"{DATASET} / {METHOD}: Graphical Lasso relationship status",
)
plt.show()


## Cross-fitted graph-guided repair experiment

The copula repair changes dependence by reordering each held-out synthetic column. Therefore, targeted, random-pair, and preserved-edge repairs retain exactly the same values in every feature column before and after repair. Only their joint arrangement changes.

In [ ]:
if experiment_cache.exists() and not FORCE_RECOMPUTE:
    with experiment_cache.open("rb") as handle:
        cached = pickle.load(handle)
    repair_results = cached["repair_results"]
    split_edge_tables = cached["split_edge_tables"]
    print(f"Loaded {experiment_cache.name}")
else:
    repair_results, split_edge_tables = run_graph_guided_repair_experiment(
        dataset_name=DATASET,
        method=METHOD,
        X_real=X_real,
        y_real=y_real,
        X_syn=X_syn,
        y_syn=y_syn,
        feature_names=feature_names,
        doses=DOSES,
        split_repeats=SPLIT_REPEATS,
        auc_repeats=AUC_REPEATS,
        discovery_fraction=DISCOVERY_FRACTION,
        changed_threshold=CHANGED_PARTIAL_THRESHOLD,
        repair_strength=REPAIR_STRENGTH,
        seed=revision_config.SEED,
    )
    with experiment_cache.open("wb") as handle:
        pickle.dump(
            {"repair_results": repair_results, "split_edge_tables": split_edge_tables},
            handle,
        )
    print(f"Wrote {experiment_cache.name}")


In [ ]:
fig, repair_summary = plot_repair_dose_response(
    repair_results,
    title=f"{DATASET} / {METHOD}: graph-guided structural repair",
)
plt.show()
display(repair_summary)


## Control and validity checks

The first check verifies that dependence repairs did not alter feature marginals. The second compares the maximum-dose targeted result against each control. A convincing result requires a larger targeted AUC reduction than all controls, not merely a lower AUC after any perturbation.

In [ ]:
copula_conditions = ["targeted", "random", "preserved_matched"]
marginal_check = (
    repair_results[repair_results["condition"].isin(copula_conditions)]
    .groupby("condition")["marginal_multiset_error"]
    .max()
    .to_frame("maximum sorted-column error")
)
display(marginal_check)
assert marginal_check["maximum sorted-column error"].fillna(0).max() < 1e-10

max_dose = max(DOSES)
max_dose_summary = repair_summary[repair_summary["dose"] == max_dose].sort_values(
    "mean_auc_reduction", ascending=False
)
display(max_dose_summary)


## Are the discovered structural errors reproducible?

A relationship is more credible when it appears among the largest discovery-set errors in several independent splits. This table is intentionally separate from RF importance: candidate relationships are selected from graph discrepancy alone.

In [ ]:
top_per_split = (
    split_edge_tables[split_edge_tables["category"].isin(ERROR_CATEGORIES)]
    .sort_values(["split", "abs_partial_error"], ascending=[True, False])
    .groupby("split", as_index=False)
    .head(max(DOSES))
)
edge_stability = (
    top_per_split.groupby(["feature_i", "feature_j"], as_index=False)
    .agg(
        selected_splits=("split", "nunique"),
        mean_partial_error=("abs_partial_error", "mean"),
        most_common_status=("category", lambda values: values.mode().iat[0]),
    )
    .sort_values(["selected_splits", "mean_partial_error"], ascending=False)
)
display(edge_stability.head(25))


## Decision rule

The hypothesis is supported for this dataset-generator pair only if:

- targeted repair produces a progressive dose-response;
- its AUC reduction exceeds random-pair and preserved-edge controls;
- marginal-only matching does not explain the same reduction;
- exact marginals are preserved by dependence repairs; and
- influential graph discrepancies recur across discovery splits.

If these conditions fail, the correct conclusion is that the tested Graphical Lasso discrepancies do not specifically explain RF separability. The remaining signal may instead be marginal, nonlinear, higher-order, or too distributed for pairwise edge repair.